# Fine-tune Pretrained Models on PHINC (Issue #4)

**Course Project - Group 8 (P08)**  
**Author:** Vinay Kumar (`Vinays-08`)

### Instructions:
1. Go to **Runtime -> Change runtime type** and select **T4 GPU** (free tier).
2. Run the cells step-by-step from top to bottom.
3. At the end, download `finetune_results.zip` containing all `results/finetune_*.json` and `results/finetune_*_errors.csv`.

## 1. Setup & Dependencies

In [ ]:
# Clone the latest repository
!git clone https://github.com/Swati89923/Translate-Roman-script-code-mixed-Hinglish-into-English.git repo
%cd repo

# Install dependencies
!pip install -q sacrebleu transformers datasets accelerate sentencepiece

## 2. Prepare PHINC Dataset & Run Pipeline Verification

In [ ]:
# Download PHINC from Zenodo and preprocess with fixed 80/10/10 split (seed 42)
!python src/download_data.py
!python src/preprocess.py

# Verify pipeline tests
!python -m pytest -q

## 3. Zero-Shot Baselines (No Fine-Tuning)
Evaluates how well pretrained models translate Roman Hinglish to English directly out of the box.

In [ ]:
# Zero-shot mT5 baseline
!python src/finetune.py --model mt5 --epochs 0

# Zero-shot mBART-50 baseline
!python src/finetune.py --model mbart --epochs 0

## 4. Fine-Tuning Pretrained Models
- **mT5-small**: 5 epochs, batch size 8, gradient accumulation 4 (effective batch 32).
- **mBART-50**: 3 epochs, batch size 8, gradient accumulation 4 with mixed precision (fp16).

In [ ]:
# Fine-tune mT5 (~15-20 minutes on T4 GPU)
!python src/finetune.py --model mt5 --epochs 5 --batch-size 8 --grad-accum 4

In [ ]:
# Fine-tune mBART-50 (~30-40 minutes on T4 GPU)
!python src/finetune.py --model mbart --epochs 3 --batch-size 8 --grad-accum 4

## 5. (Optional) IndicTrans2 Run

In [ ]:
# Uncomment to run IndicTrans2:
# !pip install -q IndicTransToolkit indic-transliteration
# !python src/finetune.py --model indictrans2 --epochs 3 --batch-size 8 --grad-accum 4

## 6. View Results & Download Artifacts

In [ ]:
import glob, json

print("=== Summary of Test Scores ===")
for path in sorted(glob.glob('results/finetune_*.json')):
    data = json.load(open(path))
    model_name = data.get('model', path)
    mode = data.get('mode', '')
    bleu = data.get('bleu', 0.0)
    chrf = data.get('chrf', 0.0)
    print(f"{model_name:30s} | Mode: {mode:11s} | BLEU: {bleu:6.2f} | chrF: {chrf:6.2f}")

# Package results and download
!zip -q -r finetune_results.zip results/finetune_*
from google.colab import files
files.download('finetune_results.zip')